# Exploration EBA Transparency Exercise — MVP Banque de France

Objectif : comprendre la structure reelle de `tr_oth.csv` (Capital, Leverage, P&L, RWAs, Assets and Liabilities)
sur les 3 annees disponibles (2023, 2024, 2025), avant de figer `load_eba.py`.

Points a verifier :
1. Le format long/tidy (LEI_Code x Period x Item) est bien stable d'une annee sur l'autre
2. Le pattern de code Item (2 premiers chiffres = annee, 5 derniers = code stable) se confirme
3. Quels `Label`/`Sheet` correspondent a CET1, ratio de levier, et qualite des actifs (NPL)
4. Verification du filtrage sur les 6 banques francaises (LEI codes)


In [1]:
import pandas as pd
from pathlib import Path
from collections import Counter

pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 100)

DATA_DIR = Path("data")
YEARS = [2023, 2024, 2025]


## 1. Chargement des 3 annees (`tr_oth.csv` uniquement pour l'instant)

In [25]:
def load_tr_oth(year: int) -> pd.DataFrame:
    path = DATA_DIR / str(year) / "tr_oth.csv"
    df = pd.read_csv(path, dtype={"LEI_Code": str, "Item": str, "Period": str})
    df["source_year"] = year
    return df

raw = {year: load_tr_oth(year) for year in YEARS}

for year, df in raw.items():
    print(f"{year} : {df.shape[0]} lignes, {df.shape[1]} colonnes")


C:\Users\iandr\AppData\Local\Temp\ipykernel_40348\1757295920.py:3: DtypeWarning: Columns (0: Footnote) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, dtype={"LEI_Code": str, "Item": str, "Period": str})
C:\Users\iandr\AppData\Local\Temp\ipykernel_40348\1757295920.py:3: DtypeWarning: Columns (0: Footnote) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, dtype={"LEI_Code": str, "Item": str, "Period": str})


2023 : 100223 lignes, 17 colonnes
2024 : 99324 lignes, 17 colonnes
2025 : 91277 lignes, 16 colonnes


In [5]:
# Colonnes identiques d'une annee a l'autre ?
for year, df in raw.items():
    print(year, list(df.columns))


2023 ['LEI_Code', 'NSA', 'Period', 'Item', 'Label', 'ASSETS_FV', 'ASSETS_Stages', 'Exposure', 'Financial_instruments', 'Amount', 'Fin_end_year', 'n_quarters', 'Footnote', 'Row', 'Column', 'Sheet', 'source_year']
2024 ['LEI_Code', 'NSA', 'Period', 'Item', 'Label', 'ASSETS_FV', 'ASSETS_Stages', 'Exposure', 'Financial_instruments', 'Amount', 'Fin_end_year', 'n_quarters', 'Footnote', 'Row', 'Column', 'Sheet', 'source_year']
2025 ['LEI_Code', 'NSA', 'Period', 'Item', 'Label', 'ASSETS_FV', 'ASSETS_Stages', 'Exposure', 'Financial_instruments', 'Amount', 'Fin_end_year', 'n_quarters', 'Row', 'Column', 'Sheet', 'source_year']


## 2. Verification du pattern de code Item

Hypothese : `Item` = 2 premiers chiffres (annee du template) + code stable de l'indicateur.
Ex: `2520140` (2025) / `2420140` (2024) / `2320140` (2023) -> meme indicateur, code stable `20140`.


In [6]:
def canonical_item(item: str) -> str:
    """Retire les 2 premiers chiffres (prefixe annee) pour obtenir le code stable."""
    return item[2:]

for year, df in raw.items():
    df["item_canonical"] = df["Item"].apply(canonical_item)

# Verification : pour un meme item_canonical, le Label est-il identique entre annees ?
labels_by_canonical = {}
for year, df in raw.items():
    for canonical, label in df[["item_canonical", "Label"]].drop_duplicates().values:
        labels_by_canonical.setdefault(canonical, {})[year] = label

# Afficher les cas ou le label differe entre annees (incoherence potentielle)
inconsistent = {
    k: v for k, v in labels_by_canonical.items()
    if len(set(v.values())) > 1
}
print(f"{len(labels_by_canonical)} codes canoniques uniques au total")
print(f"{len(inconsistent)} codes avec un Label qui differe entre annees :")
for k, v in list(inconsistent.items())[:20]:
    print(f"  {k}: {v}")


174 codes canoniques uniques au total
4 codes avec un Label qui differe entre annees :
  20116: {2023: '(-) Deductions related to assets which can alternatively be subject to a 1.250% risk weight', 2024: '(-) Deductions related to assets which can alternatively be subject to a 1.250% risk weight', 2025: '(-) Deductions related to assets which can alternatively be subject to a 1250% risk weight'}
  20117: {2023: '(-) Deductions related to assets which can alternatively be subject to a 1.250% risk weight - Of which: from securitisation positions (-)', 2024: '(-) Deductions related to assets which can alternatively be subject to a 1.250% risk weight - Of which: from securitisation positions (-)', 2025: '(-) Deductions related to assets which can alternatively be subject to a 1250% risk weight - Of which: from securitisation positions (-)'}
  20307: {2023: 'Expenses on share capital repayable on demand', 2024: 'Expenses on share capital repayable on demand', 2025: '(Expenses on share capit

## 3. Liste des `Label` / `Sheet` disponibles (pour reperer CET1, levier, NPL)

In [7]:
for year, df in raw.items():
    print(f"--- {year} ---")
    print(df[["Sheet", "Label"]].drop_duplicates().sort_values(["Sheet", "Label"]).to_string(index=False))
    print()


--- 2023 ---
      Sheet                                                                                                                                                                    Label
     Assets                                                                                              Accumulated impairment: Financial assets at amortised cost, Debt securities
     Assets                                                                                           Accumulated impairment: Financial assets at amortised cost, Loans and advances
     Assets                                                               Accumulated impairment: Financial assets at fair value through other comprehensive income, Debt securities
     Assets                                                            Accumulated impairment: Financial assets at fair value through other comprehensive income, Loans and advances
     Assets                                                                       

## 4. Filtrage sur les 6 banques francaises

LEI codes recuperes depuis la page officielle EBA (2025 EU-wide Transparency Exercise, France) :


In [8]:
FRENCH_BANKS = {
    "R0MUWSFPU8MPRO8K5P83": "BNP Paribas",
    "FR9695005MSX1OYEMGDF": "Groupe BPCE",
    "FR969500TJ5KRTCJQWXH": "Groupe Credit Agricole",
    "9695000CG7B84NLR5984": "Confederation Nationale du Credit Mutuel",
    "96950066U5XAAIRCPA78": "La Banque Postale",
    "O2RNE8IBXP4R0TD8PU41": "Societe generale S.A.",
}

for year, df in raw.items():
    present = df[df["LEI_Code"].isin(FRENCH_BANKS.keys())]["LEI_Code"].unique()
    missing = set(FRENCH_BANKS.keys()) - set(present)
    print(f"{year} : {len(present)}/6 banques francaises trouvees" + (f" -- manquantes: {missing}" if missing else ""))


2023 : 6/6 banques francaises trouvees
2024 : 6/6 banques francaises trouvees
2025 : 6/6 banques francaises trouvees


In [9]:
# Apercu des donnees filtrees pour une banque, un item, sur les 3 annees
# (a ajuster une fois qu'on a identifie le bon item_canonical pour CET1 / levier / NPL)
sample_year = 2025
df = raw[sample_year]
df_fr = df[df["LEI_Code"].isin(FRENCH_BANKS.keys())].copy()
df_fr["bank_name"] = df_fr["LEI_Code"].map(FRENCH_BANKS)
df_fr[["bank_name", "Period", "Sheet", "Label", "Amount"]].head(20)


,bank_name,Period,Sheet,Label,Amount
54945,Confederation Nationale du Credit Mutuel,202409,Capital,OWN FUNDS,77620.811636
54946,Confederation Nationale du Credit Mutuel,202409,Capital,COMMON EQUITY TIER 1 CAPITAL (net of deductions and after applying transitional adjustments),68600.201165
54947,Confederation Nationale du Credit Mutuel,202409,Capital,Capital instruments eligible as CET1 Capital (including share premium and net own capital instru...,11442.286643
54948,Confederation Nationale du Credit Mutuel,202409,Capital,Retained earnings,63903.564381
54949,Confederation Nationale du Credit Mutuel,202409,Capital,Accumulated other comprehensive income,-512.126102
54950,Confederation Nationale du Credit Mutuel,202409,Capital,Other Reserves,0.000000
54951,Confederation Nationale du Credit Mutuel,202409,Capital,Funds for general banking risk,0.000000
54952,Confederation Nationale du Credit Mutuel,202409,Capital,Minority interest given recognition in CET1 capital,0.002959
54953,Confederation Nationale du Credit Mutuel,202409,Capital,Adjustments to CET1 due to prudential filters,-459.502895
54954,Confederation Nationale du Credit Mutuel,202409,Capital,(-) Intangible assets (including Goodwill),-3837.347288


## 5. Prochaines etapes (a completer selon les resultats ci-dessus)

- [ ] Identifier precisement les `item_canonical` pour CET1 ratio, leverage ratio, NPL ratio
- [ ] Confirmer si NPL est dans `tr_oth.csv` ou s'il faut charger `tr_cre.csv`
- [ ] Verifier le chevauchement de `Period` entre fichiers annuels successifs (dedoublonnage)
- [ ] Decider de la ponderation du score composite (a discuter avant de coder `compute_score.py`)


In [10]:
# Sheets disponibles (juste les noms, pas la liste croisée avec Label)
for year, df in raw.items():
    print(year, sorted(df["Sheet"].unique()))

2023 ['Assets', 'Capital', 'Key metrics', 'Leverage', 'Liabilities', 'P&L', 'RWA OV1']
2024 ['Assets', 'Capital', 'Key metrics', 'Leverage', 'Liabilities', 'P&L', 'RWA OV1']
2025 ['Assets', 'Capital', 'Leverage', 'Liabilities', 'P&L', 'RWA OV1']


In [11]:
# Recherche des items contenant "RATIO" ou "LEVERAGE" ou "NON-PERFORMING"/"NPL"
for year, df in raw.items():
    mask = df["Label"].str.contains("RATIO|LEVERAGE|NON-PERFORMING|NPL", case=False, na=False)
    print(f"--- {year} ---")
    print(df.loc[mask, ["Sheet", "Label"]].drop_duplicates().to_string(index=False))
    print()

--- 2023 ---
      Sheet                                                                                                                                Label
    Capital                                                                             COMMON EQUITY TIER 1 CAPITAL RATIO (transitional period)
    Capital                                                                                           TIER 1 CAPITAL RATIO (transitional period)
    Capital                                                                                            TOTAL CAPITAL RATIO (transitional period)
    Capital                                                                                    COMMON EQUITY TIER 1 CAPITAL RATIO (fully loaded)
    Capital                                                                               (-) Insufficient coverage for non-performing exposures
Key metrics                                            Leverage ratio total exposure measure - using a transitional d

In [12]:
def load_tr_cre(year: int) -> pd.DataFrame:
    path = DATA_DIR / str(year) / "tr_cre.csv"
    return pd.read_csv(path, dtype={"LEI_Code": str, "Item": str, "Period": str})

raw_cre = {year: load_tr_cre(year) for year in YEARS}

for year, df in raw_cre.items():
    mask = df["Label"].str.contains("NON-PERFORMING|NPL|NPE", case=False, na=False)
    print(f"--- {year} ---")
    print(df.loc[mask, ["Sheet", "Label"]].drop_duplicates().to_string(index=False))
    print()

C:\Users\iandr\AppData\Local\Temp\ipykernel_41100\2224465638.py:3: DtypeWarning: Columns (0: Footnote) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, dtype={"LEI_Code": str, "Item": str, "Period": str})
C:\Users\iandr\AppData\Local\Temp\ipykernel_41100\2224465638.py:3: DtypeWarning: Columns (0: Footnote) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, dtype={"LEI_Code": str, "Item": str, "Period": str})


--- 2023 ---
             Sheet                                                                                                                                                       Label
Forborne exposures             Quality of forbearance Gross carrying amount on Non-performing forborne loans and advances that failed to meet the non-performing exit criteria
              NACE                                                                   Accumulated negative changes in fair value due to credit risk on non-performing exposures
               NPE                                                Collaterals and financial guarantees received on non-performing exposures on DEBT INSTRUMENTS other than HFT
               NPE                  Collaterals and financial guarantees received on non-performing exposures on Debt securities (including at amortised cost  and fair value)
               NPE               Collaterals and financial guarantees received on non-performing exposures on Lo

In [12]:
def load_tr_cre(year: int) -> pd.DataFrame:
    path = DATA_DIR / str(year) / "tr_cre.csv"
    return pd.read_csv(path, dtype={"LEI_Code": str, "Item": str, "Period": str}, low_memory=False)

raw_cre = {year: load_tr_cre(year) for year in YEARS}

# Recherche des lignes "totales" (pas les détails par collateral/type d'instrument)
for year, df in raw_cre.items():
    mask = df["Label"].str.contains(
        "Total.*non-performing|Non-performing.*Total|Gross carrying amount.*[Tt]otal|Total.*[Ll]oans and advances",
        case=False, na=False, regex=True
    )
    print(f"--- {year} ---")
    print(df.loc[mask, ["Sheet", "Label"]].drop_duplicates().to_string(index=False))
    print()

--- 2023 ---
Empty DataFrame
Columns: [Sheet, Label]
Index: []

--- 2024 ---
Empty DataFrame
Columns: [Sheet, Label]
Index: []

--- 2025 ---
Empty DataFrame
Columns: [Sheet, Label]
Index: []



In [14]:
for year, df in raw.items():  # tr_oth, pas tr_cre
    mask = df["Sheet"] == "Key metrics"
    print(f"--- {year} ---")
    print(df.loc[mask, ["Label"]].drop_duplicates().to_string(index=False))
    print()

--- 2023 ---
                                                                                                                                                                   Label
                                                                                                               Common Equity Tier 1 (CET1) capital - transitional period
                              Common Equity Tier 1 (CET1) capital - transitional period -  as if IFRS 9 or analogous ECLs transitional arrangements had not been applied
                                                                                                                                   Tier 1 capital  - transitional period
                                                  Tier 1 capital as if IFRS 9 or analogous ECLs transitional arrangements had not been applied - transitional definition
                                                                                                                                    Total capi

In [15]:
# 1. Quelles feuilles existent dans tr_cre.csv ?
for year, df in raw_cre.items():
    print(year, sorted(df["Sheet"].unique()))

2023 ['Collateral', 'Credit Risk_IRB_a', 'Credit Risk_IRB_b', 'Credit Risk_STA_a', 'Credit Risk_STA_b', 'Forborne exposures', 'NACE', 'NPE']
2024 ['Collateral', 'Credit Risk_IRB_a', 'Credit Risk_IRB_b', 'Credit Risk_STA_a', 'Credit Risk_STA_b', 'Forborne exposures', 'NACE', 'NPE']
2025 ['Collateral', 'Credit Risk_IRB_a', 'Credit Risk_IRB_b', 'Credit Risk_STA_a', 'Credit Risk_STA_b', 'Forborne exposures', 'NACE', 'NPE']


In [17]:
import pandas as pd
pd.set_option('display.max_colwidth', None) 
for year, df in raw_cre.items():
    print(f"--- {year} / NPE ---")
    print(df.loc[df["Sheet"] == "NPE", "Label"].drop_duplicates().to_string(index=False))
    print()

--- 2023 / NPE ---
                                                                                                                     Gross carrying amount on DEBT INSTRUMENTS other than HFT
                                                                                       Gross carrying amount on Debt securities (including at amortised cost  and fair value)
                                                                                    Gross carrying amount on Loans and advances (including at amortised cost  and fair value)
                                                                         Gross carrying amount on Debt securities (including at amortised cost  and fair value) - by exposure
                                                                      Gross carrying amount on Loans and advances (including at amortised cost  and fair value) - by exposure
                                                                                                               

In [18]:
# Combien de Row distincts pour un seul Label donné, sur une banque/periode ?
sample = raw_cre[2025]
one_bank_period = sample[(sample["LEI_Code"] == "O2RNE8IBXP4R0TD8PU41") & (sample["Period"] == sample["Period"].max())]
loans_rows = one_bank_period[
    (one_bank_period["Sheet"] == "NPE") &
    (one_bank_period["Label"].str.contains("Loans and advances", case=False, na=False))
]
loans_rows[["Row", "Column", "Label", "Amount"]].sort_values(["Row", "Column"])

,Row,Column,Label,Amount
585004,17,45,Gross carrying amount on Loans and advances (including at amortised cost and fair value),506510.235622
585005,17,46,Gross carrying amount on Loans and advances (including at amortised cost and fair value),492503.366999
585006,17,47,Gross carrying amount on Loans and advances (including at amortised cost and fair value),41405.305170
585007,17,48,Gross carrying amount on Loans and advances (including at amortised cost and fair value),2220.562291
585008,17,49,Gross carrying amount on Loans and advances (including at amortised cost and fair value),14006.868623
...,...,...,...,...
585234,27,54,"Accumulated impairment, accumulated changes in fair value due to credit risk and provisions on Loans and advances (including at amortised cost and fair value) - by exposure",232.992948
585235,27,55,"Accumulated impairment, accumulated changes in fair value due to credit risk and provisions on Loans and advances (including at amortised cost and fair value) - by exposure",1755.497795
585236,27,56,"Accumulated impairment, accumulated changes in fair value due to credit risk and provisions on Loans and advances (including at amortised cost and fair value) - by exposure",0.000000
585237,27,57,"Accumulated impairment, accumulated changes in fair value due to credit risk and provisions on Loans and advances (including at amortised cost and fair value) - by exposure",1755.497795


In [2]:
xls = pd.ExcelFile(DATA_DIR / "2025" / "SDD.xlsx")
print(xls.sheet_names)

['SDD']


In [4]:
sdd = pd.read_excel(DATA_DIR / "2025" / "SDD.xlsx", header=1)
print(sdd.shape)
print(sdd.columns.tolist())
sdd.head(10)

(335, 18)
['CSV', 'Template', 'Collection', 'Item', 'Item_TR_2024', 'Item_TR_2023', 'Item_TR_2022', 'Item_TR_2021', 'Item_TR_2020A', 'Item_TR_2020S', 'Item_TR_2019', 'Item_TR_2018', 'Item_TR_2017', 'Item_TR_2016', 'Item_TR_2015', 'Item_TR_2014', 'Category', 'Label']


,CSV,Template,Collection,Item,Item_TR_2024,Item_TR_2023,Item_TR_2022,Item_TR_2021,Item_TR_2020A,Item_TR_2020S,Item_TR_2019,Item_TR_2018,Item_TR_2017,Item_TR_2016,Item_TR_2015,Item_TR_2014,Category,Label
0,tr_oth.csv,Capital,TR2025,2520101,2420101.0,2320101.0,2220101.0,2120101.0,2020101.0,2020101.0,1920101.0,1820101.0,1720101.0,1620101.0,150101.0,993401.0,Capital,OWN FUNDS
1,tr_oth.csv,Capital,TR2025,2520102,2420102.0,2320102.0,2220102.0,2120102.0,2020102.0,2020102.0,1920102.0,1820102.0,1720102.0,1620102.0,150102.0,993402.0,Capital,COMMON EQUITY TIER 1 CAPITAL (net of deductions and after applying transitional adjustments)
2,tr_oth.csv,Capital,TR2025,2520103,2420103.0,2320103.0,2220103.0,2120103.0,2020103.0,2020103.0,1920103.0,1820103.0,1720103.0,1620103.0,150103.0,993403.0,Capital,Capital instruments eligible as CET1 Capital (including share premium and net own capital instru...
3,tr_oth.csv,Capital,TR2025,2520104,2420104.0,2320104.0,2220104.0,2120104.0,2020104.0,2020104.0,1920104.0,1820104.0,1720104.0,1620104.0,150104.0,993405.0,Capital,Retained earnings
4,tr_oth.csv,Capital,TR2025,2520105,2420105.0,2320105.0,2220105.0,2120105.0,2020105.0,2020105.0,1920105.0,1820105.0,1720105.0,1620105.0,150105.0,993406.0,Capital,Accumulated other comprehensive income
5,tr_oth.csv,Capital,TR2025,2520106,2420106.0,2320106.0,2220106.0,2120106.0,2020106.0,2020106.0,1920106.0,1820106.0,1720106.0,1620106.0,150106.0,993409.0,Capital,Other Reserves
6,tr_oth.csv,Capital,TR2025,2520107,2420107.0,2320107.0,2220107.0,2120107.0,2020107.0,2020107.0,1920107.0,1820107.0,1720107.0,1620107.0,150107.0,993410.0,Capital,Funds for general banking risk
7,tr_oth.csv,Capital,TR2025,2520108,2420108.0,2320108.0,2220108.0,2120108.0,2020108.0,2020108.0,1920108.0,1820108.0,1720108.0,1620108.0,150108.0,993411.0,Capital,Minority interest given recognition in CET1 capital
8,tr_oth.csv,Capital,TR2025,2520109,2420109.0,2320109.0,2220109.0,2120109.0,2020109.0,2020109.0,1920109.0,1820109.0,1720109.0,1620109.0,150109.0,993412.0,Capital,Adjustments to CET1 due to prudential filters
9,tr_oth.csv,Capital,TR2025,2520110,2420110.0,2320110.0,2220110.0,2120110.0,2020110.0,2020110.0,1920110.0,1820110.0,1720110.0,1620110.0,150110.0,993414.0,Capital,(-) Intangible assets (including Goodwill)


In [5]:
# Est-ce que tr_cre.csv apparait dans ce dictionnaire ?
print(sdd["CSV"].unique())

# Si oui, ce qu'il dit sur la feuille NPE
npe_dict = sdd[(sdd["CSV"] == "tr_cre.csv") & (sdd["Template"] == "NPE")]
npe_dict[["Item", "Category", "Label"]]

<ArrowStringArray>
['tr_oth.csv', 'tr_mrk.csv', 'tr_cre.csv', 'tr_sov.csv']
Length: 4, dtype: str


,Item,Category,Label
226,2520601,NPE,Gross carrying amount on DEBT INSTRUMENTS other than HFT
227,2520602,NPE,Gross carrying amount on Debt securities (including at amortised cost and fair value)
228,2520603,NPE,Gross carrying amount on Loans and advances (including at amortised cost and fair value)
229,2520604,NPE,Gross carrying amount on Debt securities (including at amortised cost and fair value) - by expo...
230,2520605,NPE,Gross carrying amount on Loans and advances (including at amortised cost and fair value) - by e...
231,2520606,NPE,Gross carrying amount on OFF-BALANCE SHEET EXPOSURES
232,2520608,NPE,Gross carrying amount on Cash balances at central banks and other demand deposits
233,2520611,NPE,"Accumulated impairment, accumulated changes in fair value due to credit risk and provisions on D..."
234,2520612,NPE,"Accumulated impairment, accumulated changes in fair value due to credit risk and provisions on D..."
235,2520613,NPE,"Accumulated impairment, accumulated changes in fair value due to credit risk and provisions on L..."


In [6]:
meta = pd.ExcelFile(DATA_DIR / "2025" / "TR_Metadata.xlsx")
print(meta.sheet_names)

['List of Institutions', 'Other banks', 'Dimensions used', 'Portfolio', 'Country', 'Financial_instruments', 'Exposure', 'Status', 'Perf_status', 'MKT_Modprod', 'MKT_Risk', 'Accounting_portfolio', 'Maturity', 'ASSETS_Stages', 'ASSETS_FV', 'NACE_codes', 'Fin_end_year']


In [9]:
dims = pd.read_excel(meta, sheet_name="Dimensions used", header=1)
print(dims.shape)
dims.head(20)

(15, 15)


,oth,Leverage,LEI_code,Bank_name,Period,Item,Label,ASSETS_Stages,ASSETS_FV,Exposure,Financial_instruments,Unnamed: 11,Unnamed: 12,Unnamed: 13,Amount
0,NaN,Capital,LEI_code,Bank_name,Period,Item,Label,ASSETS_Stages,ASSETS_FV,Exposure,Financial_instruments,NaN,NaN,NaN,Amount
1,NaN,RWA OV1,LEI_code,Bank_name,Period,Item,Label,ASSETS_Stages,ASSETS_FV,Exposure,Financial_instruments,NaN,NaN,NaN,Amount
2,NaN,P&L,LEI_code,Bank_name,Period,Item,Label,ASSETS_Stages,ASSETS_FV,Exposure,Financial_instruments,NaN,NaN,NaN,Amount
3,NaN,Assets,LEI_code,Bank_name,Period,Item,Label,ASSETS_Stages,ASSETS_FV,Exposure,Financial_instruments,NaN,NaN,NaN,Amount
4,NaN,Liabilities,LEI_code,Bank_name,Period,Item,Label,ASSETS_Stages,ASSETS_FV,Exposure,Financial_instruments,NaN,NaN,NaN,Amount
5,mrk,Market Risk,LEI_code,Bank_name,Period,Item,Label,Portfolio,MKT_Modprod,Mkt_risk,NaN,NaN,NaN,NaN,Amount
6,cre,Credit Risk_STA_a,LEI_code,Bank_name,Period,Item,Label,Portfolio,Country,Country_rank,Exposure,Status,Perf_Status,NACE_codes,Amount
7,NaN,Credit Risk_STA_b,LEI_code,Bank_name,Period,Item,Label,Portfolio,Country,Country_rank,Exposure,Status,Perf_Status,NACE_codes,Amount
8,NaN,Credit Risk_IRB_a,LEI_code,Bank_name,Period,Item,Label,Portfolio,Country,Country_rank,Exposure,Status,Perf_Status,NACE_codes,Amount
9,NaN,Credit Risk_IRB_b,LEI_code,Bank_name,Period,Item,Label,Portfolio,Country,Country_rank,Exposure,Status,Perf_Status,NACE_codes,Amount


In [10]:
perf = pd.read_excel(meta, sheet_name="Perf_status", header=1)
print(perf.shape)
perf

(11, 2)


,0,No breakdown by Perf_status
0,1,Performing
1,11,Performing - of which exposures with forbearance measures
2,12,Performing Of which: Instruments with significant increase in credit risk since initial recognit...
3,2,Non Performing
4,21,Non Performing Of which:\nexposures with forbearance measures
5,22,Non Performing Of which:\nUnlikely to pay that are not past-due or past-due <= 90 days
6,222,Non Performing Of which: Stage 2
7,23,NON performing - of which Stage 3
8,3,Performing but past due >30 days and <=90 days
9,4,Non Performing and Defaulted


In [13]:
print(raw_cre[2025].columns.tolist())

['LEI_Code', 'NSA', 'Period', 'Item', 'Label', 'Portfolio', 'Country', 'Country_rank', 'Exposure', 'Status', 'Perf_Status', 'NACE_codes', 'Amount', 'Row', 'Column', 'Sheet']


In [14]:
# Si Perf_Status existe comme colonne
df = raw_cre[2025]
mask_loans = df["Label"].str.contains("Gross carrying amount on Loans and advances \\(including at amortised cost", case=False, na=False, regex=True)
sample = df[mask_loans & (df["LEI_Code"] == "O2RNE8IBXP4R0TD8PU41")]
sample[["Period", "Portfolio", "Exposure", "Status", "Perf_Status", "NACE_codes", "Amount"]].head(20)

,Period,Portfolio,Exposure,Status,Perf_Status,NACE_codes,Amount
578464,202409,0,0,0,0,0,510810.840508
578465,202409,0,0,0,1,0,495743.420622
578466,202409,0,0,0,12,0,34433.639341
578467,202409,0,0,0,3,0,3758.989528
578468,202409,0,0,0,2,0,15067.419886
578469,202409,0,0,0,222,0,0.000000
578470,202409,0,0,2,4,0,15067.419886
578471,202409,0,0,0,23,0,15067.419886
578512,202409,0,101,0,0,0,19005.447752
578513,202409,0,101,0,1,0,19005.430748


In [19]:
import pandas as pd

pool = pd.read_csv(
    "data/interim/consolidated_pool.csv",
    dtype={"LEI_Code": str, "Period": str},
)

check = pool[(pool["bank_name"] == "Groupe BPCE") & (pool["Period"] == "202403")]
check[["metric", "Amount", "source_year"]]

bpce_rows = pool[(pool["Period"] == "202403") & (pool["Amount"].between(7.0e4, 7.2e4))]
for val in bpce_rows["LEI_Code"].unique():
    print(repr(val))

print()
candidates = pool[pool["LEI_Code"].str.contains("9695005", na=False)]["LEI_Code"].unique()
for val in candidates:
    print(repr(val))

'96950001WI712W7PQG45'
'A5GWLFH3KM7YV2SFQL84'
'FR9695005MSX1OYEMGDF'

'FR9695005MSX1OYEMGDF'


In [20]:
import sys
sys.path.append("..")  # si le notebook est dans training/eba/exploration/
from compute_score import load_pool, compute_ratios

pool = load_pool()
wide = compute_ratios(pool)

wide[(wide["bank_name"] == "Groupe BPCE") & (wide["Period"] == "202403")]

,LEI_Code,Period,bank_name,cet1_denominator,cet1_numerator,leverage_denominator,leverage_numerator,npl_denominator,npl_numerator,cet1_ratio,leverage_ratio,npl_ratio
44,0W2PZJM8XOY22M4GG883,202403,Groupe BPCE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
128,2138008AVF4W7FMW8W87,202403,Groupe BPCE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
212,2138009Y59EAR7H1UO97,202403,Groupe BPCE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
296,213800DBQIB6VBNU5C64,202403,Groupe BPCE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
380,213800HDJ876ACJXXD05,202403,Groupe BPCE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
10544,TO822O0VT80V06K0FH57,202403,Groupe BPCE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10628,TUKDD90GPC79G1KOE162,202403,Groupe BPCE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10712,VDYMYTQGZZ6DU0912C88,202403,Groupe BPCE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10796,VWMYAEQSTOPNV0SUGU82,202403,Groupe BPCE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
print(wide.shape)
print(wide["bank_name"].value_counts(dropna=False))

(10920, 12)
bank_name
BNP Paribas                                 1560
Confederation Nationale du Credit Mutuel    1560
Groupe BPCE                                 1560
Groupe Credit Agricole                      1560
La Banque Postale                           1560
Societe generale S.A.                       1560
NaN                                         1560
Name: count, dtype: int64


In [22]:
mask = (wide["bank_name"] == "Groupe BPCE") & (wide["Period"] == "202403")
print(mask.sum())  # nombre de lignes qui matchent reellement
wide[mask]

130


,LEI_Code,Period,bank_name,cet1_denominator,cet1_numerator,leverage_denominator,leverage_numerator,npl_denominator,npl_numerator,cet1_ratio,leverage_ratio,npl_ratio
44,0W2PZJM8XOY22M4GG883,202403,Groupe BPCE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
128,2138008AVF4W7FMW8W87,202403,Groupe BPCE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
212,2138009Y59EAR7H1UO97,202403,Groupe BPCE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
296,213800DBQIB6VBNU5C64,202403,Groupe BPCE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
380,213800HDJ876ACJXXD05,202403,Groupe BPCE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
10544,TO822O0VT80V06K0FH57,202403,Groupe BPCE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10628,TUKDD90GPC79G1KOE162,202403,Groupe BPCE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10712,VDYMYTQGZZ6DU0912C88,202403,Groupe BPCE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10796,VWMYAEQSTOPNV0SUGU82,202403,Groupe BPCE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
raw_2025 = raw["2025"] if "2025" in raw else None  # ou recharge tr_oth.csv 2025 si raw n'est plus en memoire
df = pd.read_csv("data/2025/tr_oth.csv", dtype={"LEI_Code": str, "Item": str, "Period": str}, low_memory=False)

mask = (df["Sheet"] == "Capital") & (df["Period"] == "202503") & df["Label"].str.contains("RISK EXPOSURE", case=False, na=False)
df.loc[mask, "Label"].drop_duplicates()

447                                                  TOTAL RISK EXPOSURE AMOUNT
448    TOTAL RISK EXPOSURE AMOUNT - Of which: Transitional adjustments included
461                                      TOTAL RISK EXPOSURE AMOUNT - PRE FLOOR
Name: Label, dtype: str

In [27]:
mask = (df["Sheet"] == "Capital") & (df["Period"] == "202503") & (df["Label"] == "TOTAL RISK EXPOSURE AMOUNT")
df.loc[mask, ["LEI_Code", "Amount"]].head(10)

,LEI_Code,Amount
447,0W2PZJM8XOY22M4GG883,28815.090686
1271,2138008AVF4W7FMW8W87,5269.936234
2095,2138009Y59EAR7H1UO97,2568.365028
2919,213800DBQIB6VBNU5C64,30398.910335
3743,213800HDJ876ACJXXD05,6062.664739
4567,213800TC9PZRBHMJW403,1195.564579
5391,213800X3Q9LSAKRUWY91,123005.280639
6215,222100K6QL2V4MLHWQ08,21013.802191
7039,254900RNFMDM0P11YR84,13605.288567
7615,2G5BKIC2CB69PRJH1W31,34179.549069


In [28]:
mask = (df["Sheet"] == "Capital") & (df["Period"] == "202503") & (df["Label"] == "TOTAL RISK EXPOSURE AMOUNT")
french_leis = ["R0MUWSFPU8MPRO8K5P83", "FR9695005MSX1OYEMGDF", "FR969500TJ5KRTCJQWXH",
               "9695000CG7B84NLR5984", "96950066U5XAAIRCPA78", "O2RNE8IBXP4R0TD8PU41"]

df.loc[mask & df["LEI_Code"].isin(french_leis), ["LEI_Code", "Amount"]]

,LEI_Code,Amount
55392,9695000CG7B84NLR5984,361596.779312
56216,96950066U5XAAIRCPA78,96705.852119
68656,FR9695005MSX1OYEMGDF,451453.070433
69480,FR969500TJ5KRTCJQWXH,640578.251584
83156,O2RNE8IBXP4R0TD8PU41,393071.785958
85628,R0MUWSFPU8MPRO8K5P83,783440.361050


In [29]:
pool = pd.read_csv("data/interim/consolidated_pool.csv", dtype={"LEI_Code": str, "Period": str})

check = pool[(pool["LEI_Code"] == "O2RNE8IBXP4R0TD8PU41") & (pool["Period"] == "202503")]
check[["metric", "Amount", "source_year"]]

,metric,Amount,source_year
7560,cet1_denominator,393071.785958,2025
7561,cet1_numerator,NaN,2025
7562,leverage_denominator,NaN,2025
7563,leverage_numerator,NaN,2025
7564,npl_denominator,506448.149030,2025
7565,npl_numerator,14254.431956,2025


In [30]:
df = pd.read_csv("data/2025/tr_oth.csv", dtype={"LEI_Code": str, "Item": str, "Period": str}, low_memory=False)

mask = (df["LEI_Code"] == "O2RNE8IBXP4R0TD8PU41") & (df["Period"] == "202503")

# CET1 numerateur : tout ce qui contient "COMMON EQUITY TIER 1" sur la feuille Capital
print(df.loc[mask & (df["Sheet"] == "Capital") & df["Label"].str.contains("COMMON EQUITY TIER 1", case=False, na=False), ["Label", "Amount"]])
print()

# Leverage num + denom : tout le contenu de la feuille Leverage pour cette banque/periode
print(df.loc[mask & (df["Sheet"] == "Leverage"), ["Label", "Amount"]])

                                                                                              Label  \
83120  COMMON EQUITY TIER 1 CAPITAL (net of deductions and after applying transitional adjustments)   
83158                                      COMMON EQUITY TIER 1 CAPITAL RATIO (transitional period)   
83161                                                   COMMON EQUITY TIER 1 CAPITAL (fully loaded)   
83162                                             COMMON EQUITY TIER 1 CAPITAL RATIO (fully loaded)   
83171                          COMMON EQUITY TIER 1 CAPITAL RATIO (transitional period - pre floor)   

             Amount  
83120  51890.082368  
83158      0.132012  
83161           NaN  
83162           NaN  
83171      0.132012  

                                                                                       Label  \
83236                 Tier 1 capital - transitional definition (numerator of Leverage ratio)   
83237                                            Tier 1 